In [1]:
# Cell 1 - Imports
import os
from dotenv import load_dotenv
from llama_index.core import StorageContext, load_index_from_storage
from llama_index.core import Settings

import sys
# setting path
# Aggiungi la cartella PARENT della cartella Parla_con_PG_TM
project_root = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# import from parent directory llm.py:
from Parla_con_PG_TM.llm import init_local_embed_model
Settings.embed_model = init_local_embed_model()

load_dotenv()

True

In [2]:
# Cell 3 - Load existing vector store
import chromadb
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, Settings, StorageContext, load_index_from_storage
PERSIST_DIR = project_root + "\Parla_con_PG_TM\chroma_db"

def load_vector_store():
    """Load the existing vector store from disk"""
    if not os.path.exists(PERSIST_DIR):
        raise ValueError(f"Storage directory '{PERSIST_DIR}' not found")
    Settings.embed_model = init_local_embed_model()
    print("Loading vector store...")
    chroma_client = chromadb.PersistentClient(path=PERSIST_DIR)
    collection = chroma_client.get_collection(name="ardania_chat_memory")
    vector_store = ChromaVectorStore(chroma_collection=collection)
    index = VectorStoreIndex.from_vector_store(vector_store=vector_store)
    print("Vector store loaded successfully")
    return index

In [3]:

# Cell 4 - Query function
def query_similar_docs(index, query_text, top_k=3):
    """
    Retrieve top k most similar documents to the query
    """
    # Create query embedding and retrieve similar docs
    Settings.embed_model = init_local_embed_model()
    retriever = index.as_retriever(similarity_top_k=top_k)
    nodes = retriever.retrieve(query_text)
    
    print(f"\nTop {top_k} similar documents to '{query_text}':\n")
    for i, node in enumerate(nodes, 1):
        print(f"Document {i}:")
        print(f"Score: {node.score:.4f}")
        print(f"Content: {node.text}\n")
    
    return nodes

In [4]:
# Cell 5 - Execute query
if __name__ == "__main__":
    # Load index
    index = load_vector_store()
    
    # Query for similar documents
    query_text = "non ne sono a conoscenza"
    similar_docs = query_similar_docs(index, query_text)

Loading vector store...
Vector store loaded successfully

Top 3 similar documents to 'non ne sono a conoscenza':

Document 1:
Score: 0.0000
Content: User Message: e invece non conosci Amon?

                         Assistant Response: Non ne sono a conoscenza, messere.

Document 2:
Score: 0.0000
Content: User Message: <debug> cosa sai di Amon?

                         Assistant Response: Non ne sono a conoscenza, messere.

Document 3:
Score: 0.0000
Content: User Message: in che città vivi?

                         Assistant Response: Non ne sono a conoscenza, messere.



In [16]:
# print all the scores of similar documents
print("\nScores of similar documents:")
for i, doc in enumerate(similar_docs, 1):
    print(f"Document {i} Score: {doc.score:.4f}")
# End of script


Scores of similar documents:
Document 1 Score: 0.0000
Document 2 Score: 0.0000
Document 3 Score: 0.0000


In [ ]:
similar_docs

[NodeWithScore(node=TextNode(id_='753b6b53b379287030ada5bcefe9e2732b7f4dba300fc19f5cbda4ea85e22524', embedding=None, metadata={'turn_number': 1, 'timestamp': '2025-09-16T23:19:46.846304', 'message_type': 'dialogue', 'hash': '753b6b53b379287030ada5bcefe9e2732b7f4dba300fc19f5cbda4ea85e22524', 'user_message': 'e invece non conosci Amon?', 'assistant_response': 'Non ne sono a conoscenza, messere.', 'importance_score': 0.5, 'memory_type': 'chat'}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=[], relationships={}, metadata_template='{key}: {value}', metadata_separator='\n', text='User Message: e invece non conosci Amon?\n\n                         Assistant Response: Non ne sono a conoscenza, messere.', mimetype='text/plain', start_char_idx=None, end_char_idx=None, metadata_seperator='\n', text_template='{metadata_str}\n\n{content}'), score=0.0),
 NodeWithScore(node=TextNode(id_='267558c6f86ff55df18804188e951121894d54a53665749d64ece58107e54faa', embedding=None, metadata={'memo

In [14]:
# list all nodes in chromadb
chroma_client = chromadb.PersistentClient(path=PERSIST_DIR)
collection = chroma_client.get_collection(name="ardania_lore")
all_nodes = collection.get(include=['embeddings', 'documents', 'metadatas'])
all_nodes

{'ids': ['...',
  ...],
 'embeddings': array([[-10.67116356,  14.4089756 ,  15.15435028, ...,   1.47767496,
          -1.16503453,  -1.37726378],
        [ -6.64227772,  10.09934616,   1.66720438, ...,   0.16000199,
          -3.61995745,  -8.62080002],
        [-10.44360828,  11.25726891,  10.92274666, ...,  -2.30848455,
          -3.52803659,  -7.58520317],
        ...,
        [-10.15772152,   1.81719065,   4.39070797, ...,   4.85802317,
          -2.49662375,  -4.65943003],
        [ -9.0923748 ,   3.98379874,   1.66084385, ...,  -0.28674078,
           1.35653472,  -2.67056894],
        [-15.46280766,   8.1502409 ,   3.29914236, ...,  -6.13024664,
           0.93511415,  -7.72105408]]),
 'documents': ['CHAPTERONELA GEOGRAFIA DI ARDANIA1.1 Continente Umano1.1.1 Descrizione generaleL’intero reame, compreso tra l’isola di Hammerheim e i monti Orquirian, tra la catena montuosa degli Orus Maer e leTerre Selvagge, viene chiamato comunemente “Reame Umano”.Con questo nome si è soliti defi

In [15]:
#take first node from all_nodes and print its metadata and document
first_node = all_nodes['documents'][0]

# print the embedding of the first node
first_node_embedding = all_nodes['embeddings']
print("Embedding:", first_node_embedding)

Embedding: [[-10.67116356  14.4089756   15.15435028 ...   1.47767496  -1.16503453
   -1.37726378]
 [ -6.64227772  10.09934616   1.66720438 ...   0.16000199  -3.61995745
   -8.62080002]
 [-10.44360828  11.25726891  10.92274666 ...  -2.30848455  -3.52803659
   -7.58520317]
 ...
 [-10.15772152   1.81719065   4.39070797 ...   4.85802317  -2.49662375
   -4.65943003]
 [ -9.0923748    3.98379874   1.66084385 ...  -0.28674078   1.35653472
   -2.67056894]
 [-15.46280766   8.1502409    3.29914236 ...  -6.13024664   0.93511415
   -7.72105408]]


In [17]:
import chromadb

# 1. Usa lo stesso modello
embed_model = init_local_embed_model()
query_text = "Hammerheim"
query_embedding = embed_model.get_text_embedding(query_text)

# 2. Interroga ChromaDB direttamente
PERSIST_DIR = project_root + "\Parla_con_PG_TM\chroma_db"
chroma_client = chromadb.PersistentClient(path=PERSIST_DIR)
collection = chroma_client.get_collection(name="ardania_lore")

results = collection.query(
    query_embeddings=[query_embedding], # Nota: va passato come lista di liste
    n_results=5
)

# 3. Controlla il risultato
# Chroma restituisce le 'distances', non la 'similarity'. 
# Valori più bassi sono migliori.
print(results['distances'])

[[559699.5, 562340.0, 562704.8125, 562936.8125, 563111.125]]


In [5]:
import chromadb

# ... tuo codice per inizializzare il client ...
chroma_client = chromadb.PersistentClient(path=PERSIST_DIR)
collection = chroma_client.get_collection(name="ardania_chat_memory")

# Stampa i metadati della collezione, inclusa la funzione di distanza
# Di default è 'l2' (distanza euclidea)
print(collection.metadata)

{'description': 'Memoria delle conversazioni con il personaggio'}


In [7]:
import chromadb
from llama_index.core import Settings
from llama_index.core.vector_stores import VectorStoreQuery # Importa VectorStoreQuery
from llama_index.vector_stores.chroma import ChromaVectorStore

# --- Configurazione (come già hai) ---
Settings.embed_model = init_local_embed_model()
PERSIST_DIR = project_root + "\Parla_con_PG_TM\chroma_db"
top_k = 5
query_text = "Hammerheim"

# --- Client e Collezione ---
chroma_client = chromadb.PersistentClient(path=PERSIST_DIR)
collection = chroma_client.get_collection(name="ardania_lore")
vector_store = ChromaVectorStore(chroma_collection=collection)

# --- DEBUG: Interroga direttamente il Vector Store ---

# 1. Ottieni l'embedding della query ESATTAMENTE come farebbe LlamaIndex
query_embedding = Settings.embed_model.get_text_embedding(query_text)
print(f"Embedding della query (prima 5 dimensioni): {query_embedding[:5]}...")

# 2. Costruisci un oggetto di query per il vector store
# Questo è l'oggetto che il retriever costruisce internamente
vector_store_query = VectorStoreQuery(
    query_embedding=query_embedding,
    similarity_top_k=top_k
)

# 3. Esegui la query direttamente sul wrapper
query_result = vector_store.query(vector_store_query)

# --- Analisi del Risultato ---
print("--- Analisi del VectorStoreQueryResult ---")
print(f"Nodi trovati: {len(query_result.nodes)}")
print(f"Similarità trovate: {query_result.similarities}")
print(f"ID trovati: {query_result.ids}")

# Ispeziona i singoli nodi e i loro score
if query_result.nodes:
    for node, similarity in zip(query_result.nodes, query_result.similarities):
        # NOTA: il nodo stesso ha un suo campo .score che potrebbe essere ancora vuoto
        # La fonte della verità qui è la lista query_result.similarities
        print("-" * 20)
        print(f"Similarity dalla lista: {similarity:.4f}")
        print(f"Score dentro al nodo: {node.score}") 
        print(f"Testo: {node.get_content()[:100]}...")

Embedding della query (prima 5 dimensioni): [-157.72824096679688, -12.835587501525879, 19.125015258789062, 1.9681577682495117, 20.47868537902832]...
--- Analisi del VectorStoreQueryResult ---
Nodi trovati: 5
Similarità trovate: [0.0, 0.0, 0.0, 0.0, 0.0]
ID trovati: ['8b390f928c38b1b00308c74e316a47b5b1b3f5b35df97a0a923594d93935a7fe', 'c130e15d7e5226f55301a5df8d1d3b108f7d27e09021387643f636691e5c6564', 'd3e65ee6637ef710010778b51aba612e9da7b322dd8b47dd317b0f3cdf29d8af', 'a922147e48bcc1f9a7ccaceb47c6bb8aac4131b698bb79c222926247059ec557', 'd0d747f6a3e91a12eda07bc5dbdcc9c7c37f19b69cb1cd2c77b4f6abb273bbe7']
--------------------
Similarity dalla lista: 0.0000


AttributeError: 'TextNode' object has no attribute 'score'

In [8]:
# (Mantieni tutto il setup precedente: import, Settings, client, ecc.)
import json

# 1. Ottieni l'embedding ESATTAMENTE come farebbe LlamaIndex
query_embedding = Settings.embed_model.get_text_embedding(query_text)

# 2. Interroga ChromaDB direttamente, chiedendo tutti i dati
# Questo simula la chiamata che ChromaVectorStore fa internamente
results = collection.query(
    query_embeddings=[query_embedding],
    n_results=top_k,
    include=["metadatas", "documents", "distances"] # Chiediamo esplicitamente tutto
)

# 3. Stampa il risultato GREZZO ricevuto da ChromaDB
print("--- Risultato Grezzo da ChromaDB ---")
print(json.dumps(results, indent=2, ensure_ascii=False))

--- Risultato Grezzo da ChromaDB ---
{
  "ids": [
    [
      "8b390f928c38b1b00308c74e316a47b5b1b3f5b35df97a0a923594d93935a7fe",
      "c130e15d7e5226f55301a5df8d1d3b108f7d27e09021387643f636691e5c6564",
      "d3e65ee6637ef710010778b51aba612e9da7b322dd8b47dd317b0f3cdf29d8af",
      "a922147e48bcc1f9a7ccaceb47c6bb8aac4131b698bb79c222926247059ec557",
      "d0d747f6a3e91a12eda07bc5dbdcc9c7c37f19b69cb1cd2c77b4f6abb273bbe7"
    ]
  ],
  "embeddings": null,
  "documents": [
    [
      "nno vissuto nelle vicinanze del mare imparando a temere e rispettare il suoregno. Chi prende il mare sa che deve ossequiare e temere la dea, non mancherà mai farle un dono simbolico prima di sal-pare e di ringraziarla una volta approdato. I fedeli di Danu sono spesso persone volubili e legate alla propria indipendenza.Per loro, il diritto alla propria libertà viene prima di ogni cosa e il mare, enorme e sconfinato, li rappresenta. Combat-tono con passione ogni forma di oppressione e sottomission",
      "li